# JavaScript::D3 subs

Anton Antonov  
September 2026

---

## Introduction

This notebook extract signatures of implementation subs of "JavaScript::D3" and associates with he top-level subs (e.g. `js-d3-list-plot`).
Then exports to a JSON file that can be used in AI agent skills.

---

## Setup

In [3]:
use Data::Importers;
use Data::Reshapers;
use Data::Summarizers;
use Hash::Merge;
use JSON::Fast;

---

## File URLs and content

In [4]:
my $base-url = 'https://raw.githubusercontent.com/antononcube/Raku-JavaScript-D3/refs/heads/main/lib/JavaScript/D3';

my @file-urls = <Charts Chess Gauge Graph Images Plots Plots3D Random>.map({ "{$base-url}/{$_}.rakumod" });
@file-urls.elems

8

In [5]:
my %source-codes = @file-urls.map({ $_.split('/', :skip-empty).tail.split('.').head => data-import($_, 'asis') });
 
deduce-type(%source-codes)


Assoc(Atom((Str)), Atom((Str)), 8)

In [ ]:
my %sub-defs = %source-codes.map({
    $_.key => 
    do with $_.value.match(/ ['multi sub' | 'our multi'] \h+ $<sub-name>=(\w+) \h* '(' $<sub-args>=(.*?) ')' \h+ '{' /):g {
        $/.map({ <sub-name sub-args> Z=> $_<sub-name sub-args>».Str })».Hash 
    }
})

In [8]:
my @dsSubDefs = %sub-defs.kv.map( -> $k, @v { @v.map({ %(basename => $k, |$_) }) }).flat(1);
deduce-type(@dsSubDefs)

Vector(Assoc(Atom((Str)), Atom((Str)), 3), 61)

In [9]:
#% html
@dsSubDefs.pick(6)
==> to-html(field-names => <basename sub-name sub-args>)

basename,sub-name,sub-args
Chess,Chessboard,"Str $data, *%args"
Plots,ListPlotGeneric,"$data where is-positional-of-lists($data, 2), *%args"
Plots,DateListPlot,"$data where is-positional-of-str-date-time-value-lists($data), *%args"
Charts,BarChart,"$data where $data ~~ (Array | List | Seq) && $data.all ~~ Pair:D, *%args"
Charts,BubbleChart,"$data where is-positional-of-lists($data, 4), *%args"
Plots,ListLinePlot,"$data, *%args"


---

## Delegations

In [61]:
my $url = 'https://raw.githubusercontent.com/antononcube/Raku-JavaScript-D3/refs/heads/main/lib/JavaScript/D3.rakumod';
my $top-code = data-import($url, 'asis');
text-stats($top-code)

(chars => 30673 words => 3052 lines => 732)

In [62]:
my %delegations = $top-code.split('#|').map({

    do with $_ ~~ / $<top>=('js-d3-' [\w | '-']+) .*? $<to>=('JavaScript::D3::' \w+ '::' \w+) / {
        $/<top>.Str => $/<to>.Str
    }
});

%delegations<js-d3-list-plot> = 'JavaScript::D3::Plots::ListPlotGeneric';
%delegations<js-d3-list-line-plot> = 'JavaScript::D3::Plots::ListPlotGeneric';

JavaScript::D3::Plots::ListPlotGeneric

Verify 1-to-1 correspondence:

In [63]:
%delegations.map({ %(top => $_.key, to => $_.value )}).List
andthen group-by($_, <top>)
andthen $_».elems
andthen $_.values.minmax

1..1

In [64]:
my %delegations-inv = %delegations.invert;

deduce-type(%delegations-inv)

Assoc(Atom((Str)), Atom((Str)), 21)

----

## Re-map

In [66]:
my @dsTopSubArgs = 
    @dsSubDefs.map( -> %rec { 
        my $key = %delegations-inv.keys.first({ $_.ends-with(%rec<sub-name>) });
        if $key {
            merge-hash(%rec, %(top => %delegations-inv{$key}))
        } else { Empty }
    });

@dsTopSubArgs .= sort({ [|$_<top sub-name>, $_<sub-args>.elems] });

deduce-type(@dsTopSubArgs)


Vector(Assoc(Atom((Str)), Atom((Str)), 4), 60)

In [67]:
#% html
@dsTopSubArgs[40..48]
==> to-html(field-names => <top sub-name sub-args basename>)

top,sub-name,sub-args,basename
js-d3-heatmap-plot,HeatmapPlot,"$data where is-positional-of-lists($data, 2), *%args",Plots
js-d3-heatmap-plot,HeatmapPlot,"@data is copy where @data.all ~~ Map, :$width is copy = 600, :$height is copy = 600, Str :plot-label(:$title) = '', UInt :plot-label-font-size(:$title-font-size) = 16, Str :plot-label-color(:$title-color) = 'Black', Str :x-label(:$x-axis-label) = '', :x-label-color(:$x-axis-label-color) is copy = Whatever, :x-label-font-size(:$x-axis-label-font-size) is copy = Whatever, Str :y-label(:$y-axis-label) = '', :y-label-color(:$y-axis-label-color) is copy = Whatever, :y-label-font-size(:$y-axis-label-font-size) is copy = Whatever, :$color = Whatever, Str :$color-palette = 'Inferno', Str :$background = 'White', Str :$tick-labels-color = 'Black', :$tick-labels-font-size is copy = Whatever, Str:D :$tick-labels-font-family = 'Helvetica', Numeric :$opacity = 0.7, Str :$plot-labels-color = 'Black', :$plot-labels-font-size is copy = Whatever, Str :$plot-labels-font-family = 'Courier', :$x-tick-labels is copy = Whatever, :$y-tick-labels is copy = Whatever, Bool :$sort-tick-labels = True, Bool :$show-groups = True, :$low-value is copy = Whatever, :$high-value is copy = Whatever, :$margins is copy = Whatever, Bool :$tooltip = True, :$tooltip-background-color is copy = Whatever, :$tooltip-color is copy = Whatever, :$mesh = True, :$grid-lines is copy = Whatever, :$round-corners is copy = Whatever, Str :$format = 'jupyter', :$div-id = Whatever",Plots
js-d3-histogram,Histogram,"$data, UInt $number-of-bins, *%args",Charts
js-d3-histogram,Histogram,"$data where $data ~~ Seq, *%args",Charts
js-d3-histogram,Histogram,"@data where @data.all ~~ Numeric, UInt :bins(:$number-of-bins) = 20, Str :$background= 'white', Str :$color= 'steelblue', Str :$color-scheme = 'schemeSet2', :$width = 600, :$height = 400, Str :plot-label(:$title) = '', UInt :plot-label-font-size(:$title-font-size) = 16, Str :plot-label-color(:$title-color) = 'Black', Str :x-label(:$x-axis-label) = '', :x-label-color(:$x-axis-label-color) is copy = Whatever, :x-label-font-size(:$x-axis-label-font-size) is copy = Whatever, Str :y-label(:$y-axis-label) = '', :y-label-color(:$y-axis-label-color) is copy = Whatever, :y-label-font-size(:$y-axis-label-font-size) is copy = Whatever, :$grid-lines is copy = False, :$margins is copy = Whatever, Str :$format = 'jupyter', :$div-id = Whatever",Charts
js-d3-image,Image,"$data where $data ~~ Seq, *%args",Images
js-d3-image,Image,"@data where is-matrix(@data, Numeric:D), Str :$color-palette= ""Greys"", :$width is copy = Whatever, :$height is copy = Whatever, :$low-value is copy = Whatever, :$high-value is copy = Whatever, Str :$format = 'jupyter', :$div-id = Whatever",Images
js-d3-image-display,ImageDisplay,"$spec, :$width is copy = Whatever, :$height is copy = Whatever, Str :$format = 'jupyter', :$div-id = Whatever",Images
js-d3-list-line-plot,ListPlotGeneric,"$data where $data ~~ Seq, *%args",Plots


---

## Export

In [68]:
select-columns(@dsTopSubArgs, <top sub-args>)
==> { rename-columns($_, {top => 'name', 'sub-args' => 'signature'}) }()
==> to-json(:sorted-keys)
==> { spurt( $*CWD.add(<.. skills raku-javascript-d3-plots-and-charts references subs.json>), $_ )}()

True